# 01 - Data Collection with the SpaceX REST API
**IBM Applied Data Science Capstone - SpaceX Falcon 9 First-Stage Landing Prediction**

Author: Pranav Vyankatesh Jagdale

**Goal:** request past launches from the SpaceX API, resolve the rocket / launchpad / payload / core IDs, and build a Falcon 9 launch table.

> Run every cell in order. Keep your own outputs - they feed your final slides.

In [21]:
import datetime
from functools import lru_cache
import requests
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

## 1. Request the launch list

In [22]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"

try:
    response = requests.get(spacex_url)
    print("status:", response.status_code)
    response.raise_for_status()
    data = pd.json_normalize(response.json())
    print("Successfully fetched data from live SpaceX API.")
except Exception as e:
    print(f"Live API request failed ({e}). Falling back to static backup dataset...")
    # Using the correct static CSV backup URL from the IBM course resources
    fallback_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
    data = pd.read_csv(fallback_url)
    print("Successfully fetched backup dataset.")

data.head(3)

status: 525
Live API request failed (525 Server Error: <none> for url: https://api.spacexdata.com/v4/launches/past). Falling back to static backup dataset...
Successfully fetched backup dataset.


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857


## 2. Keep only the columns we need, single-core / single-payload launches, and the course date cut-off

In [23]:
# If we fell back to the pre-processed CSV dataset, skip the raw JSON mapping steps
if 'FlightNumber' in data.columns:
    print("Pre-processed backup dataset detected. Skipping API raw mapping steps.")
    # Ensure required columns for the rest of the notebook are present and named correctly
    data['date'] = pd.to_datetime(data['Date']).dt.date
    data = data[data['date'] <= datetime.date(2020, 11, 13)]
    print(data.shape)
else:
    data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
    data = data[data['cores'].map(len) == 1]
    data = data[data['payloads'].map(len) == 1]
    data['cores'] = data['cores'].map(lambda x: x[0])
    data['payloads'] = data['payloads'].map(lambda x: x[0])
    data['date'] = pd.to_datetime(data['date_utc']).dt.date
    data = data[data['date'] <= datetime.date(2020, 11, 13)]
    print(data.shape)

data.head()

Pre-processed backup dataset detected. Skipping API raw mapping steps.
(90, 18)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,date
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,2010-06-04
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,2012-05-22
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,2013-03-01
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,2013-09-29
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,2013-12-03


## 3. Helper functions that resolve IDs (cached so each ID is requested once)

In [24]:
# If we fell back to the pre-processed CSV dataset, skip the API lookups entirely
if 'FlightNumber' in data.columns:
    print("Pre-processed backup dataset detected. Skipping API ID-resolution lookups.")
else:
    @lru_cache(maxsize=None)
    def get_json(endpoint, item_id):
        r = requests.get(f"https://api.spacexdata.com/v4/{endpoint}/{item_id}")
        r.raise_for_status()
        return r.json()

    BoosterVersion, PayloadMass, Orbit = [], [], []
    LaunchSite, Longitude, Latitude = [], [], []
    Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
    LandingPad, Block, ReusedCount, Serial = [], [], [], []

    for _, row in data.iterrows():
        BoosterVersion.append(get_json("rockets", row["rocket"])["name"])
        pad = get_json("launchpads", row["launchpad"])
        LaunchSite.append(pad["name"])
        Longitude.append(pad["longitude"])
        Latitude.append(pad["latitude"])
        pl = get_json("payloads", row["payloads"])
        PayloadMass.append(pl["mass_kg"])
        Orbit.append(pl["orbit"])
        core = row["cores"]
        Outcome.append(f"{core["landing_success"]} {core["landing_type"]}")
        Flights.append(core["flight"])
        GridFins.append(core["gridfins"])
        Reused.append(core["reused"])
        Legs.append(core["legs"])
        LandingPad.append(core["landpad"])
        if core["core"] is not None:
            c = get_json("cores", core["core"])
            Block.append(c["block"])
            ReusedCount.append(c["reuse_count"])
            Serial.append(c["serial"])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
    print("lookups done")

Pre-processed backup dataset detected. Skipping API ID-resolution lookups.


## 4. Assemble the dataset

In [25]:
# Use the existing data if we are using the fallback backup dataset
if "FlightNumber" in data.columns:
    df = data.copy()
    print("Using already assembled backup dataset.")
else:
    launch_dict = {
        "FlightNumber": list(data["flight_number"]),
        "Date": list(data["date"]),
        "BoosterVersion": BoosterVersion,
        "PayloadMass": PayloadMass,
        "Orbit": Orbit,
        "LaunchSite": LaunchSite,
        "Outcome": Outcome,
        "Flights": Flights,
        "GridFins": GridFins,
        "Reused": Reused,
        "Legs": Legs,
        "LandingPad": LandingPad,
        "Block": Block,
        "ReusedCount": ReusedCount,
        "Serial": Serial,
        "Longitude": Longitude,
        "Latitude": Latitude,
    }
    df = pd.DataFrame(launch_dict)

df.head()

Using already assembled backup dataset.


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,date
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,2010-06-04
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,2012-05-22
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,2013-03-01
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,2013-09-29
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,2013-12-03


## 5. Keep Falcon 9 only, fix the flight numbers, and handle missing payload mass

In [26]:
data_falcon9 = df[df["BoosterVersion"] != "Falcon 1"].copy()
data_falcon9.loc[:, "FlightNumber"] = list(range(1, data_falcon9.shape[0] + 1))
print(data_falcon9.isnull().sum())
mean_mass = data_falcon9["PayloadMass"].mean()
data_falcon9["PayloadMass"] = data_falcon9["PayloadMass"].replace(
    np.nan, mean_mass
)
print("mean payload mass used:", round(mean_mass, 1))
data_falcon9.to_csv("my_dataset_part_1.csv", index=False)
print("Successfully created my_dataset_part_1.csv!")
data_falcon9.shape

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
date               0
dtype: int64
mean payload mass used: 6105.0
Successfully created my_dataset_part_1.csv!


(90, 18)

## 6. Write your own summary here
- How many launches did you collect?
- Which columns have missing values and what did you do about them?
- Anything that surprised you?